### Function Calling with OpenAI API
https://platform.openai.com/docs/guides/function-calling?api-mode=chat

In [38]:
from openai import OpenAI
import json
from dotenv import load_dotenv
load_dotenv()

client = OpenAI()

tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current temperature for provided coordinates in celsius.",
        "parameters": {
            "type": "object",
            "properties": {
                "latitude": {"type": "number"},
                "longitude": {"type": "number"}
            },
            "required": ["latitude", "longitude"],
            "additionalProperties": False
        },
        "strict": True
    }
}]

messages = [{"role": "user", "content": "What's the weather like in Wuhan today?"}]

completion = client.chat.completions.create(
    model="gpt-4.1",
    messages=messages,
    tools=tools,
)
print(completion.choices[0].message)
print(completion.choices[0].message.content)
print(completion.choices[0].message.tool_calls)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_Vf44aRD0vwqdOOJ95X8WgVFG', function=Function(arguments='{"latitude":30.5928,"longitude":114.3055}', name='get_weather'), type='function')])
None
[ChatCompletionMessageToolCall(id='call_Vf44aRD0vwqdOOJ95X8WgVFG', function=Function(arguments='{"latitude":30.5928,"longitude":114.3055}', name='get_weather'), type='function')]


In [39]:
import requests

def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']

In [45]:
tool_call = completion.choices[0].message.tool_calls[0]
args = json.loads(tool_call.function.arguments)

result = get_weather(args["latitude"], args["longitude"])
print(result)

29.1


In [46]:
messages.append(completion.choices[0].message)  # append model's function call message
messages.append({                               # append result message
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": str(result)
})

completion_2 = client.chat.completions.create(
    model="gpt-4.1",
    messages=messages,
    tools=tools,
)
completion_2.choices[0].message.content

'Today in Wuhan, the current temperature is approximately 29.1°C. If you’d like more detailed weather information (like humidity, wind, or forecast), let me know!'